In [ ]:
import numpy as np, pandas as pd, scipy.sparse as sp, time
from sklearn.decomposition import TruncatedSVD, PCA
from sklearn.preprocessing import normalize


In [4]:
#DE1: load both sheets (check the column names first!)
sheets = pd.read_excel("data/raw/online_retail_II.xlsx", sheet_name=None)
df = pd.concat(sheets.values(), ignore_index=True)
print(df.columns.tolist(), len(df))



['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country'] 1067371


In [ ]:
#DE2: Clean: remove cancellations, non-positive quantity or price, non-product codes, lines without a customer ID. Log how many rows each rule removed
print(f"Rows before cleaning: {len(df)}")

cancellation_mask = df["Invoice"].astype(str).str.startswith("C", na=False)
print(f"Cancellation invoices: {cancellation_mask.sum()}")
df = df[~cancellation_mask]  # remove cancellations
print(f"Rows after removing cancellations: {len(df)}")

non_positive_mask = (df["Quantity"] <= 0) | (df["Price"] <= 0)
print(f"Non-positive quantity or price rows: {non_positive_mask.sum()}")
df = df[~non_positive_mask]  # remove non-positive quantity or price
print(f"Rows after removing non-positive quantity or price: {len(df)}")

non_product_code_mask = df["StockCode"].isin(["POST", "M", "D", "BANK CHARGES"])
print(f"Non-product code rows: {non_product_code_mask.sum()}")
df = df[~non_product_code_mask]  # remove non-product codes
print(f"Rows after removing non-product codes: {len(df)}")

no_customer_id_mask = df["Customer ID"].isna() | (df["Customer ID"] == "")
print(f"Rows without a customer ID: {no_customer_id_mask.sum()}")
df = df[~no_customer_id_mask]  # remove lines without a customer ID
print(f"Rows after removing lines without a customer ID: {len(df)}")


Rows before cleaning: 802965


AttributeError: 'Series' object has no attribute 'startswith'

In [ ]:
# DE4: sparse customer x product matrix from TRAINING rows only
cust = {c: i for i, c in enumerate(train["Customer ID"].unique())}
prod = {p: j for j, p in enumerate(train["StockCode"].unique())}
rows = train["Customer ID"].map(cust); cols = train["StockCode"].map(prod)
vals = np.log1p(train["Quantity"]) # one possible purchase value
A = sp.csr_matrix((vals, (rows, cols)), shape=(len(cust), len(prod)))
print(A.shape, A.nnz, f"sparsity {1 - A.nnz / (A.shape[0] * A.shape[1]):.2%}")